# Step 1: SBERT Embedding Generation

**환경**: 로컬 GPU (RTX 5060 Ti) · Google Colab 호환  
**입력**: CSV 파일 (텍스트 컬럼 포함)  
**출력**: `data/embeddings/<timestamp>_embeddings.npy` + `_metadata.csv`

---
### 체크리스트
- [ ] `config.yaml`에 `data.input_path`, `data.text_column` 설정 완료
- [ ] GPU 사용 시: CUDA 드라이버 설치 확인 (`nvidia-smi`)
- [ ] Colab 사용 시: 런타임 → GPU 선택 후 실행

## 0. 환경 설정

In [ ]:
# Colab에서 실행 시 — Google Drive 마운트
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # 프로젝트 경로를 Google Drive 기준으로 수정하세요
    PROJECT_DIR = '/content/drive/MyDrive/korean-bertopic-pipeline'
else:
    from pathlib import Path
    PROJECT_DIR = str(Path('..').resolve())  # notebooks/ 한 단계 위

print(f'Project: {PROJECT_DIR}')

In [ ]:
# 패키지 설치 (Colab / 초기 설치 시)
# !pip install -q sentence-transformers bertopic umap-learn hdbscan kiwipiepy pyyaml tqdm

# GPU 확인
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    DEVICE = 'cuda'
else:
    print('GPU 미감지 — CPU 모드')
    DEVICE = 'cpu'

## 1. 설정

In [ ]:
import os, sys
sys.path.insert(0, PROJECT_DIR)

from pipeline.config import load_config

# config.yaml을 프로젝트 루트에서 로드
cfg = load_config(os.path.join(PROJECT_DIR, 'config.yaml'))
data_cfg  = cfg['data']
embed_cfg = cfg['embedding']

# ── 직접 수정 가능 (config.yaml 보다 우선) ──────────────────────────
MODEL_CHOICE = 'bge-m3'   # 'bge-m3' | 'ko-sroberta'

MODEL_CONFIGS = {
    'bge-m3': {
        'name': 'BAAI/bge-m3',
        'max_chars': 2500,
        'batch_gpu': 4,
        'batch_cpu': 8,
        'dim': 1024,
    },
    'ko-sroberta': {
        'name': 'jhgan/ko-sroberta-multitask',
        'max_chars': 1000,
        'batch_gpu': 16,
        'batch_cpu': 32,
        'dim': 768,
    },
}

mcfg = MODEL_CONFIGS[MODEL_CHOICE]
MODEL_NAME = mcfg['name']
MAX_CHARS  = mcfg['max_chars']
BATCH_SIZE = mcfg['batch_gpu'] if DEVICE == 'cuda' else mcfg['batch_cpu']

INPUT_PATH  = os.path.join(PROJECT_DIR, data_cfg['input_path'])
OUTPUT_DIR  = os.path.join(PROJECT_DIR, embed_cfg.get('output_dir', 'data/embeddings'))
TEXT_COL    = data_cfg.get('text_column', 'text')

print(f'Model:   {MODEL_NAME}')
print(f'Device:  {DEVICE} | batch={BATCH_SIZE} | max_chars={MAX_CHARS}')
print(f'Input:   {INPUT_PATH}')
print(f'Output:  {OUTPUT_DIR}')

## 2. 데이터 로드

In [ ]:
import pandas as pd

df = pd.read_csv(INPUT_PATH, encoding='utf-8-sig')
print(f'전체: {len(df):,}행 | 컬럼: {list(df.columns)}')

# 텍스트 컬럼 자동 탐색
for col in [TEXT_COL, 'text', 'abstract', 'content', '본문']:
    if col in df.columns:
        TEXT_COL = col
        break
print(f'텍스트 컬럼: {TEXT_COL!r}')

# 선택적: 검증 필터
verify_col = data_cfg.get('verify_column')
verify_vals = data_cfg.get('verify_values')
if verify_col and verify_col in df.columns and verify_vals:
    before = len(df)
    df = df[df[verify_col].isin(verify_vals)].copy()
    print(f'필터 ({verify_col}): {before:,} → {len(df):,}')

# null 제거
before = len(df)
df = df[df[TEXT_COL].notna() & (df[TEXT_COL].str.len() > 0)].copy()
if before != len(df):
    print(f'null 제거: {before:,} → {len(df):,}')

# 임베딩용 절단 (원본 유지)
df['text_for_embed'] = df[TEXT_COL].str.slice(0, MAX_CHARS)

n_truncated = (df[TEXT_COL].str.len() > MAX_CHARS).sum()
print(f'\n유효 문서: {len(df):,}건')
print(f'절단 대상: {n_truncated:,}건 ({n_truncated/len(df)*100:.1f}%)')
df.head(2)

## 3. 임베딩 생성

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np, time

print(f'모델 로딩: {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
print(f'차원: {model.get_sentence_embedding_dimension()} | 장치: {model.device}')

In [ ]:
docs = df['text_for_embed'].tolist()

t0 = time.time()
embeddings = model.encode(
    docs,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)
elapsed = time.time() - t0

print(f'완료: {elapsed:.0f}초 ({elapsed/60:.1f}분)')
print(f'속도: {len(docs)/elapsed:.1f} docs/s')
print(f'Shape: {embeddings.shape}')

## 4. 저장 및 검증

In [ ]:
import os, json
os.makedirs(OUTPUT_DIR, exist_ok=True)

ts = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

embed_path = os.path.join(OUTPUT_DIR, f'{ts}_embeddings.npy')
np.save(embed_path, embeddings)

meta_path = os.path.join(OUTPUT_DIR, f'{ts}_metadata.csv')
df.to_csv(meta_path, index=False, encoding='utf-8-sig')

info = {
    'timestamp': ts, 'model_name': MODEL_NAME,
    'embedding_dim': int(embeddings.shape[1]),
    'n_documents': int(embeddings.shape[0]),
    'max_chars': MAX_CHARS, 'batch_size': BATCH_SIZE,
    'device': DEVICE, 'normalize_embeddings': True,
    'elapsed_seconds': round(elapsed, 1),
}
with open(os.path.join(OUTPUT_DIR, f'{ts}_embedding_info.json'), 'w') as f:
    json.dump(info, f, indent=2)

print(f'임베딩: {embed_path}')
print(f'메타: {meta_path}')

# 검증
loaded = np.load(embed_path)
assert loaded.shape == embeddings.shape
print(f'검증 통과 ✓ ({loaded.shape})')

## 5. 품질 확인 (선택)

코사인 유사도가 **0.2~0.7** 범위이면 정상입니다.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

rng = np.random.default_rng(42)
idx = rng.choice(len(embeddings), size=min(10, len(embeddings)), replace=False)
sim = cosine_similarity(embeddings[idx])
np.fill_diagonal(sim, np.nan)
vals = sim[~np.isnan(sim)]
print(f'코사인 유사도 — 평균: {vals.mean():.3f} | 최소: {vals.min():.3f} | 최대: {vals.max():.3f}')

---
## 다음 단계

```bash
# 파라미터 튜닝 (선택, ~30분)
uv run python scripts/02_tune.py

# BERTopic 모델링
uv run python scripts/03_model.py
```

또는 **`notebooks/02_bertopic.ipynb`** 실행